# CTDDG conditional fine-tuning on **RF-labelled BindingDB**

**Directly compatible with `CTDDG_pretraining_final(2).ipynb`**, not the older pasted `02_finetune_ctddg` notebook. The original graph traversal, action encoding, GCN/GRU/policy class implementations and state-dict names are copied from that exact pretrainer; we add CDGCN protein conditioning at **(1)** the first-atom distribution and **(2)** every GCN layer. Only those new parameters may be missing during weight transfer.

**Pipeline:** `BindingDB_Data_Preprocessing.ipynb` → original `01_generate_prose_embeddings.ipynb` outputs (if needed) → this notebook. Protein embeddings are frozen, offline ProSE/Bepler HDF5 vectors indexed by verified FASTA order (`p1..pN`, 6165 dimensions).

**Safeguards:** no training until `RUN_TRAINING=True`; original fold is retained; validation is disjoint at the **protein level**; test set is never used in fitting/early stopping; ChEMBL anti-leakage gate; strict provenance and checkpoint shape checks; tiny CPU sanity tests; step-0 diagnostics; finite-loss and gradient checks; separate class metrics; deterministic epoch batching; **atomic latest/best checkpoints, backup, RNG+optimizer+mid-epoch resume and signal-aware save**.

**Important:** The published Eq. 8 signed toxic log-likelihood objective is mathematically unbounded. A falling negative training loss is not evidence of improved molecules. No training or molecule-generation results are claimed by this notebook. Validation monitors non-toxic likelihood; final assessment must involve actually generated molecules, RF/independent toxicity scoring and binding evaluation. The paper's BindingDB labels were pkCSM, while these are your selected RF predictions.


In [ ]:
from pathlib import Path
from collections import Counter
import os, json, hashlib, math, time, random, signal, warnings, shutil
import numpy as np, pandas as pd, networkx as nx
from scipy import sparse
from rdkit import Chem, rdBase
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F

REPO_ROOT=Path(os.environ.get('CTDDG_REPO_ROOT','/home/muhamed/repo/CTDDG'))
DATA_DIR=REPO_ROOT/'datasets/processed/ctddg_bindingdb_rf_d1_v1'
PRETRAIN_RUN_DIR=REPO_ROOT/'REWORKED_CODE/pretraining/pretraining_runs/ctddg_torch_eq8_bs32_v1'
PRETRAIN_CHECKPOINT=PRETRAIN_RUN_DIR/'checkpoint.pt'
PRETRAIN_META=PRETRAIN_RUN_DIR/'run_meta.json'
# MUST be the exact vocabulary used in that stage-1 checkpoint.
ATOM_TYPES_FILE=REPO_ROOT/'REWORKED_CODE/chembl_preprocessed/atom_types.txt'
RUN_DIR=REPO_ROOT/'REWORKED_CODE/finetuning/finetuning_runs/ctddg_bindingdb_rf_d1_v1'
RUN_TRAINING=False  # SAFE DEFAULT: preflight and step-0 diagnostics only
AUTO_RESUME=True    # resume ckpt_last.pt with model + optimizer + RNG + epoch/batch position
RESTART_WITH_NEW_DIR=False # if changing experiment, use a *new RUN_DIR*, never overwrite it
REQUIRE_VERIFIED_ZERO_LEAKAGE=True
VERIFY_DEPENDENCY_HASHES=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED=42
BATCH_SIZE=8             # stage-2 CDGCN/CTDDG paper: 8; separate from stage-1's 32
K_PATHS=5
ALPHA=0.8
TOX_LAMBDA=1.0
OBJECTIVE='ctddg_eq8'    # 'ctddg_eq8' (paper Eq 8) or 'cdgcn_nll' (true all-positive control)
MAX_EPOCHS=10
PATIENCE=3
MIN_DELTA=0.01
LEARNING_RATE=1e-4
LR_DECAY=0.002
LR_DECAY_STEP=100
GRAD_CLIP=3.0         # elementwise, as in your pretraining notebook
LOG_EVERY=100
SAVE_EVERY=200
VAL_EVERY_EPOCHS=1
VAL_MAX_PAIRS=None     # None => full protein-held-out validation; a number => fast diagnostic subset
SMOKE_MAX_TRAIN_PAIRS=None # e.g. 64 for smoke ONLY; changes run signature and output folder!
# If a class log-likelihood becomes pathologically negative, stop safely rather than call it success.
# This is a heuristic alert, NOT a mathematical stability fix or a claim of paper equivalence.
TOX_LL_WARNING_THRESHOLD=-500.0
STOP_ON_TOX_LL_WARNING=False
MAX_ITER_PER_MOL=120
RECEPTIVE_FIELD=2
N_C=6165

random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
assert OBJECTIVE in {'ctddg_eq8','cdgcn_nll'}
assert BATCH_SIZE>=2 and K_PATHS==5 and abs(ALPHA-0.8)<1e-9
assert 0 <= TOX_LAMBDA and MAX_EPOCHS>0 and PATIENCE>0
print('Torch',torch.__version__,'RDKit',rdBase.rdkitVersion,'device',DEVICE)
print('Pretrained checkpoint:',PRETRAIN_CHECKPOINT,'exists?',PRETRAIN_CHECKPOINT.exists())
print('Prepared BindingDB:',DATA_DIR,'exists?',DATA_DIR.exists())
print('Run folder:',RUN_DIR)


## 1. Verify labelled dataset, fold membership, FASTA and embedding provenance

The preprocessing notebook owns all split decisions. No split is re-created or re-shuffled here. A changed TSV, vocabulary, HDF5 or selected RF is fatal. The stage-1 metadata's `atom_types_sha256` must equal the current ordered file, and checkpoint dimensions must equal the exact pretraining model configuration.


In [ ]:
def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as g:
        for x in iter(lambda:g.read(1<<20),b''):h.update(x)
    return h.hexdigest()

for x in (DATA_DIR/'dataset_manifest.json',ATOM_TYPES_FILE,PRETRAIN_CHECKPOINT,PRETRAIN_META):
    if not x.is_file():raise FileNotFoundError(f'Missing required input: {x}')
manifest=json.loads((DATA_DIR/'dataset_manifest.json').read_text())
pre_meta=json.loads(PRETRAIN_META.read_text())
if manifest.get('schema_version')!=1:raise RuntimeError('Unsupported BindingDB manifest')
if REQUIRE_VERIFIED_ZERO_LEAKAGE and manifest.get('pretrain_leakage',{}).get('status')!='verified_zero':
    raise RuntimeError('Stage-1 ligand exclusion NOT verified. Preprocess using the exact ChEMBL training input, examine overlap and regenerate a clean stage-1 model if needed. Do not set this gate off for a reproduction claim.')
# The exact ChEMBL input checked for BindingDB overlap must ALSO be the exact
# pretraining input whose checksum stage 1 recorded. A zero-overlap check on
# a different file says nothing about the weights being transferred.
leakage=manifest['pretrain_leakage']
if leakage.get('status')=='verified_zero':
    stage1_chembl_hash=pre_meta.get('chembl_sha256')
    verified_chembl_hash=leakage.get('pretrain_sha256')
    if not stage1_chembl_hash or stage1_chembl_hash!=verified_chembl_hash:
        raise RuntimeError('The file verified for ligand leakage is NOT provably the ChEMBL dataset in stage-1 run_meta.json. Recheck the actual pretraining input and rerun preprocessing. No provenance override is available.')
    chembl_path=Path(leakage['pretrain_file'])
    if not chembl_path.is_file() or file_hash(chembl_path)!=verified_chembl_hash:
        raise RuntimeError('The leakage-checked ChEMBL file has moved/changed; cannot verify stage-1 provenance.')
if pre_meta.get('atom_types_sha256')!=file_hash(ATOM_TYPES_FILE):
    raise RuntimeError('Atom-vocabulary hash does not match pretraining: do not replace/reorder atom types after training.')
if file_hash(ATOM_TYPES_FILE)!=manifest['dependencies_sha256'].get(str(ATOM_TYPES_FILE)):
    raise RuntimeError('BindingDB preparation used a DIFFERENT ordered atom vocabulary')
if VERIFY_DEPENDENCY_HASHES:
    for name,expected in manifest['dependencies_sha256'].items():
        path=Path(name)
        if not path.is_file() or file_hash(path)!=expected:
            raise RuntimeError(f'Preprocessed input changed/missing since preparation: {path}')
    for name,expected in manifest['output_sha256'].items():
        if file_hash(DATA_DIR/name)!=expected:
            raise RuntimeError(f'Output altered after dataset preparation: {name}')

splits={part:pd.read_csv(DATA_DIR/f'{part}.tsv',sep='	') for part in ('train','val','test')}
for part,df in splits.items():
    required={'smiles','protein_index','protein_id','ames'}
    if not required.issubset(df.columns):raise RuntimeError(f'{part} missing {required-set(df.columns)}')
    if df.empty or df[['smiles','protein_index','ames']].isna().any().any():raise ValueError(f'{part} missing/empty data')
    if not set(df['ames'].astype(int).unique()).issubset({0,1}):raise ValueError(f'{part} has labels outside 0/1')
    if any(row.protein_id!=f'p{int(row.protein_index)+1}' for row in df.itertuples()):raise ValueError('Protein id/index mismatch')
    print(f'{part}: {len(df):,} pairs, {df.protein_index.nunique()} proteins, RF Ames-positive {df.ames.mean():.1%}')
trrows=set(splits['train'].protein_index.astype(int));vrows=set(splits['val'].protein_index.astype(int))
if trrows&vrows:raise RuntimeError('Train and validation share a protein')

def load_prose_matrix(part):
    paths=manifest['protein_paths'][part]
    h5=Path(paths['h5']);prot=Path(paths['list']);fa=Path(paths['fasta'])
    seqs=[''.join(x.split()).upper() for x in prot.read_text().splitlines() if x.strip()]
    # Need exactly p1..pN and exactly the original protein-list order.
    records=[];name=None;buf=[]
    for line in fa.read_text().splitlines():
        if line.startswith('>'):
            if name is not None:records.append((name,''.join(buf).upper()))
            name=line[1:].split()[0];buf=[]
        elif line.strip():buf.append(''.join(line.split()))
    if name is not None:records.append((name,''.join(buf).upper()))
    if len(records)!=len(seqs) or any(n!=f'p{i+1}' or s!=seqs[i] for i,(n,s) in enumerate(records)):
        raise RuntimeError('FASTA order differs from authoritative protein list')
    with h5py.File(h5,'r') as reader:
        keys={f'p{i}' for i in range(1,len(seqs)+1)}
        if set(reader.keys())!=keys:raise ValueError('HDF5 must contain keys p1..pN exactly')
        vec=[]
        for i in range(1,len(seqs)+1):
            x=np.asarray(reader[f'p{i}'],dtype=np.float32)
            if x.shape!=(N_C,) or not np.isfinite(x).all():raise ValueError(f'Invalid embedding p{i}: {x.shape}')
            vec.append(x)
    return np.stack(vec),seqs

C_train,train_sequences=load_prose_matrix('train')
# We do NOT load held-out TEST embeddings during optimisation. Their hashes and order were verified above.
assert max(max(trrows),max(vrows))<len(C_train)
print('Train/validation ProSE:',C_train.shape,'all values finite')


## 2. Graph-path encoding copied from **your actual stage-1 notebook**

The next cells copy the model's atom-type mapping, graph traversal, action expansion, path proposal, group/edge construction, and original PyTorch GCN/GRU/action policy. Any change to those would invalidate transfer, even if tensor shapes still matched. One extra `mol_ids_rep` tensor is carried by the conditional collator to add the right protein embedding to every node.


In [ ]:
class MoleculeSpec:
    def __init__(self, file_name):
        rows = []
        with Path(file_name).open(encoding='utf8') as f:
            for i,line in enumerate(f,1):
                if not line.strip(): continue
                parts = line.strip().split(',')
                if len(parts)!=3: raise ValueError(f'{file_name}:{i}: expected symbol,charge,explicitHs')
                rows.append((parts[0], int(parts[1]), int(parts[2])))
        if not rows or len(rows)!=len(set(rows)):raise ValueError('Empty or duplicated atom vocabulary')
        self.atom_types=rows
        self.atom_index={t:i for i,t in enumerate(rows)}
        self.bond_orders=[Chem.BondType.AROMATIC,Chem.BondType.SINGLE,Chem.BondType.DOUBLE,Chem.BondType.TRIPLE]
        self.bond_index={b:i for i,b in enumerate(self.bond_orders)}
        self.num_atom_types=len(self.atom_types)
        self.num_bond_types=len(self.bond_orders)
        self.max_iter=MAX_ITER_PER_MOL

    def get_atom_type(self, a):
        key=(a.GetSymbol(), a.GetFormalCharge(), a.GetNumExplicitHs())
        if key not in self.atom_index:raise ValueError(f'Out-of-vocabulary atom {key}; check atom_types.txt')
        return self.atom_index[key]

    def get_bond_type(self, b):
        bt=b.GetBondType()
        if bt not in self.bond_index:raise ValueError(f'Unsupported bond {bt}')
        return self.bond_index[bt]

    def index_to_atom(self, idx):
        symbol,charge,hs=self.atom_types[int(idx)]
        atom=Chem.Atom(symbol);atom.SetFormalCharge(charge);atom.SetNumExplicitHs(hs)
        return atom

    def index_to_bond(self, mol, begin_id, end_id, idx):
        mol.AddBond(int(begin_id),int(end_id),self.bond_orders[int(idx)])

_mol_spec=None
def get_mol_spec():
    global _mol_spec
    if _mol_spec is None:_mol_spec=MoleculeSpec(ATOM_TYPES_FILE)
    return _mol_spec

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for c in iter(lambda:f.read(1<<20),b''):h.update(c)
    return h.hexdigest()


print("Stage-1 atom-vocabulary width:",get_mol_spec().num_atom_types)

In [ ]:
def get_graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f'RDKit cannot parse {smiles!r}')
    if mol.GetNumAtoms() < 1:
        raise ValueError(f'No atoms: {smiles!r}')
    # build graph
    atom_types, atom_ranks, bonds, bond_types = [], [], [], []
    for a, r in zip(mol.GetAtoms(), Chem.CanonicalRankAtoms(mol)):
        atom_types.append(get_mol_spec().get_atom_type(a))
        atom_ranks.append(r)
    for b in mol.GetBonds():
        idx_1, idx_2, bt = b.GetBeginAtomIdx(), b.GetEndAtomIdx(), get_mol_spec().get_bond_type(b)
        bonds.append([idx_1, idx_2])
        bond_types.append(bt)

    # build nx graph
    graph = nx.Graph()
    graph.add_nodes_from(range(len(atom_types)))
    graph.add_edges_from(bonds)

    if not nx.is_connected(graph):
        raise ValueError(f'Disconnected fragments: {smiles!r}')
    return graph, atom_types, atom_ranks, bonds, bond_types


def get_graph_from_smiles_list(smiles_list):
    graph_list = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)

        # build graph
        atom_types, bonds, bond_types = [], [], []
        for a in mol.GetAtoms():
            atom_types.append(get_mol_spec().get_atom_type(a))
        for b in mol.GetBonds():
            idx_1, idx_2, bt = b.GetBeginAtomIdx(), b.GetEndAtomIdx(), get_mol_spec().get_bond_type(b)
            bonds.append([idx_1, idx_2])
            bond_types.append(bt)

        X_0 = np.array(atom_types, dtype=np.int32)
        A_0 = np.concatenate([np.array(bonds, dtype=np.int32),
                              np.array(bond_types, dtype=np.int32)[:, np.newaxis]],
                             axis=1)
        graph_list.append([X_0, A_0])
    return graph_list


def traverse_graph(graph, atom_ranks, current_node=None, step_ids=None, p=0.9, log_p=0.0):
    if current_node is None:
        #print(f"Traversal has started")
        next_nodes = range(len(atom_ranks))
        step_ids = [-1, ] * len(next_nodes)
        next_node_ranks = atom_ranks
    else:
        next_nodes = graph.neighbors(current_node)  # get neighbor nodes
        next_nodes = [n for n in next_nodes if step_ids[n] < 0] # filter visited nodes
        next_node_ranks = [atom_ranks[n] for n in next_nodes] # get ranks for neighbors
    next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda _x:_x[1])] # sort by rank

    # iterate through neighbors
    while len(next_nodes) > 0:
        if len(next_nodes)==1:
            next_node = next_nodes[0]
        elif random.random() >= (1 - p):
            next_node = next_nodes[0]
            log_p += np.log(p)
        else:
            next_node = next_nodes[random.randint(1, len(next_nodes) - 1)]
            log_p += np.log((1.0 - p) / (len(next_nodes) - 1))
        step_ids[next_node] = max(step_ids) + 1
        _, log_p = traverse_graph(graph, atom_ranks, next_node, step_ids, p, log_p)
        next_nodes = [n for n in next_nodes if step_ids[n] < 0] # filter visited nodes


    return step_ids, log_p


def single_reorder(X_0, A_0, step_ids):

    X_0, A_0 = np.copy(X_0), np.copy(A_0)


    # Singleton graphs contain zero bonds: INIT(atom) -> END.
    # Maintain a [0,3] bond array for downstream graph operations.
    if A_0.shape[0] == 0:
        if len(X_0) != 1:
            raise ValueError('A bondless graph must have exactly one atom')
        return X_0, np.empty((0,3), dtype=np.int32)

    step_ids = np.array(step_ids, dtype=np.int32)

    # sort by step_ids
    sorted_ids = np.argsort(step_ids)
    X_0 = X_0[sorted_ids]
    A_0[:, 0], A_0[:, 1] = step_ids[A_0[:, 0]], step_ids[A_0[:, 1]]
    # max_b is the max_index of each bond,min_b is the min index of each index
    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    A_0 = A_0[np.lexsort([-min_b, max_b]), :]

    # separate append and connect
    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    is_append = np.concatenate([np.array([True]), max_b[1:] > max_b[:-1]])
    A_0 = np.concatenate([np.where(is_append[:, np.newaxis],
                                 np.stack([min_b, max_b], axis=1),
                                 np.stack([max_b, min_b], axis=1)),
                        A_0[:, -1:]], axis=1)

    return X_0, A_0


def single_expand(X_0, A_0):
    #print(f"We are in single expand function")
    X_0, A_0 = np.copy(X_0), np.copy(A_0)

    # expand X

    is_append_iter = np.less(A_0[:, 0], A_0[:, 1]).astype(np.int32)


    NX = np.cumsum(np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1))
    shift = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    X_index = np.arange(NX.sum(), dtype=np.int32) - np.repeat(shift, NX)


    X = X_0[X_index]

    _, A_index = np.tril_indices(A_0.shape[0])

    A = A_0[A_index, :]

    NA = np.arange(A_0.shape[0] + 1)                  # Num_of_bonds + 1

    action_type = 1 - is_append_iter
    atom_type = np.where(action_type == 0, X_0[A_0[:, 1]], 0)
    bond_type = A_0[:, 2]
    append_pos = np.where(action_type == 0, A_0[:, 0], 0)
    connect_pos = np.where(action_type == 1, A_0[:, 1], 0)
    actions = np.stack([action_type, atom_type, bond_type, append_pos, connect_pos],
                       axis=1)
    last_action = [[2, 0, 0, 0, 0]]
    actions = np.append(actions, last_action, axis=0)
    #print(f"actions is {actions}")
    action_0 = np.array([X_0[0]], dtype=np.int32)
    #print(f"action_0 is {action_0}")


    last_atom_index = shift + NX - 1
    last_atom_mask = np.zeros_like(X)
    last_atom_mask[last_atom_index] = np.where(
        np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1) == 1,
        np.ones_like(last_atom_index),
        np.ones_like(last_atom_index) * 2)


    return action_0, X, NX, A, NA, actions, last_atom_mask


def get_d(A, X):

    _to_sparse = lambda _A, _X: sparse.coo_matrix((np.ones([_A.shape[0] * 2], dtype=np.int32),
                                                   (np.concatenate([_A[:, 0], _A[:, 1]], axis=0),
                                                    np.concatenate([_A[:, 1], _A[:, 0]], axis=0))),
                                                    shape=[_X.shape[0], ] * 2)   # this is creating a square matrtix of size len(X)


    A_sparse = _to_sparse(A, X)

    d2 = A_sparse * A_sparse
    d3 = d2 * A_sparse

    D_2 = np.stack(d2.nonzero(), axis=1)
    D_2 = D_2[D_2[:, 0] < D_2[:, 1], :]

    D_3 = np.stack(d3.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]

    D_3_sparse = _to_sparse(D_3, X)
    D_3_sparse = D_3_sparse - D_3_sparse.multiply(A_sparse)
    D_3 = np.stack(D_3_sparse.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]

    return D_2, D_3


def merge_single_0(X_0, A_0, NX_0, NA_0):

    cumsum = np.cumsum(np.pad(NX_0, [[1, 0]], mode='constant')[:-1])

    A_0[:, :2] += np.stack([np.repeat(cumsum, NA_0), ] * 2, axis=1)

    D_0_2, D_0_3 = get_d(A_0, X_0)

    A_split = []
    for i in range(get_mol_spec().num_bond_types):
        A_i = A_0[A_0[:, 2] == i, :2]
        A_split.append(A_i)
    A_split.extend([D_0_2, D_0_3])
    A_0 = A_split


    NX_rep_0 = np.repeat(np.arange(NX_0.shape[0]), NX_0)


    return X_0, A_0, NX_0, NX_rep_0


def merge_single(X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p):

    X, A, NX, NX_rep = merge_single_0(X, A, NX, NA)
    cumsum = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    actions[:, -2] += cumsum * (actions[:, 0] == 0)
    actions[:, -1] += cumsum * (actions[:, 0] == 1)
    mol_ids_rep = np.repeat(mol_ids, NX)
    rep_ids_rep = np.repeat(rep_ids, NX)

    return X, A,\
           mol_ids_rep, rep_ids_rep, iw_ids,\
           last_append_mask,\
           NX, NX_rep,\
           action_0, actions, \
           log_p


def process_single(record, k, p):
    """Return k graph paths but exactly ONE molecular toxicity label.

    Label is attached to the *molecule*, not to the importance-sampling paths.
    Format of input: '<SMILES> <0|1>'.
    """
    try:
        smiles, label_raw = record.strip().rsplit(maxsplit=1)
    except ValueError as exc:
        raise ValueError(f"Expected '<SMILES> <0|1>', got {record!r}") from exc
    if label_raw not in ('0','1'):
        raise ValueError(f'Invalid Ames label {label_raw!r} in {record[:120]!r}')
    smiles_class = int(label_raw)
    graph, atom_types, atom_ranks, bonds, bond_types = get_graph_from_smiles(smiles)
    if not nx.is_connected(graph):
        raise ValueError(f'Disconnected SMILES (remove salts/fragments first): {smiles}')
    X_0 = np.asarray(atom_types, dtype=np.int32)
    A_0 = np.column_stack((np.asarray(bonds, dtype=np.int32).reshape(-1,2), np.asarray(bond_types, dtype=np.int32)))
    if len(bonds) + 1 > get_mol_spec().max_iter:
        raise ValueError(f'{smiles}: requires {len(bonds)+1} generation steps, max={get_mol_spec().max_iter}')

    X, A, NX, NA = [], [], [], []
    mol_ids, rep_ids, iw_ids = [], [], []
    action_0, actions, last_append_mask, log_p = [], [], [], []
    for path_idx in range(k):
        steps, lp = traverse_graph(graph, atom_ranks, p=p)
        X_i, A_i = single_reorder(X_0, A_0, steps)
        init_i, X_i, NX_i, A_i, NA_i, actions_i, mask_i = single_expand(X_i, A_i)
        X.append(X_i); A.append(A_i); NX.append(NX_i); NA.append(NA_i)
        action_0.append(init_i); actions.append(actions_i); last_append_mask.append(mask_i)
        mol_ids.append(np.zeros_like(NX_i, dtype=np.int32))
        rep_ids.append(np.full_like(NX_i, path_idx, dtype=np.int32))
        iw_ids.append(np.full_like(NX_i, path_idx, dtype=np.int32))
        log_p.append(lp)

    return (np.concatenate(X), np.concatenate(A), np.concatenate(NX), np.concatenate(NA),
            np.concatenate(mol_ids), np.concatenate(rep_ids), np.concatenate(iw_ids),
            np.concatenate(action_0), np.concatenate(actions), np.concatenate(last_append_mask),
            np.asarray(log_p, dtype=np.float32),
            np.asarray([smiles_class], dtype=np.int32))


def get_mol_from_graph(X, A, sanitize=True):
    try:
        mol = Chem.RWMol(Chem.Mol())

        X, A = X.tolist(), A.tolist()
        for i, atom_type in enumerate(X):
            mol.AddAtom(get_mol_spec().index_to_atom(atom_type))

        for atom_id1, atom_id2, bond_type in A:
            get_mol_spec().index_to_bond(mol, atom_id1, atom_id2, bond_type)
    except:
        return None

    if sanitize:
        try:
            mol = mol.GetMol()
            Chem.SanitizeMol(mol)
            return mol
        except:
            return None
    else:
        return mol


def get_mol_from_graph_list(graph_list, sanitize=True):
    mol_list = [get_mol_from_graph(X, A, sanitize) for X, A in graph_list]
    return mol_list

In [ ]:
def collate_graph_records(records, k=K_PATHS, alpha=ALPHA, device=DEVICE):
    # Reuse original MXNet graph construction and importance-sampled paths.
    fields=[[] for _ in range(12)]
    mol_id_offset, path_id_offset = 0,0
    nsteps_per_molecule=[]
    for i,rec in enumerate(records):
        rec_parts=list(process_single(rec,k,alpha))
        rec_parts[4] += i   # molecule IDs
        rec_parts[6] += i*k # unique path IDs across molecules
        for j,a in enumerate(rec_parts):fields[j].append(a)
        mol=Chem.MolFromSmiles(rec.rsplit(maxsplit=1)[0])
        nsteps_per_molecule.append(mol.GetNumBonds()+1)
    X,A,NX,NA,mol_ids,rep_ids,iw_ids,action_0,actions,last_mask,log_p,tox = [
        np.concatenate(col,axis=0) for col in fields
    ]
    X, A, mol_ids_rep,rep_ids_rep,iw_ids,last_mask,NX,NX_rep,action_0,actions,log_p = merge_single(
        X,A,NX,NA,mol_ids,rep_ids,iw_ids,action_0,actions,last_mask,log_p)

    # The original loader has one RNN step for every intermediate graph; graph-to-RNN
    # is [B,K,120] padded with zeros and rnn-to-graph indexes only real steps.
    B=len(records)
    graph_to_rnn=np.zeros([B,k,get_mol_spec().max_iter],dtype=np.int64)
    rnn_to_graph=[]
    curr=0
    for i,steps in enumerate(nsteps_per_molecule):
        if steps>get_mol_spec().max_iter:raise ValueError(f'Graph requires {steps} steps; max={get_mol_spec().max_iter}')
        for path in range(k):
            graph_to_rnn[i,path,:steps]=np.arange(steps)+curr
            rnn_to_graph += [(i,path,t) for t in range(steps)]
            curr += steps
    assert curr==len(NX)
    rnn_to_graph=np.asarray(rnn_to_graph,dtype=np.int64)
    edges=[]
    for adj in A:
        if len(adj)==0:
            edges.append(torch.empty((2,0),dtype=torch.long,device=device))
        else:
            both=np.concatenate([adj[:,0:2],adj[:,[1,0]]],axis=0)
            edges.append(torch.as_tensor(both.T.copy(),dtype=torch.long,device=device))
    batch={
      'X':torch.as_tensor(X.astype(np.int64),device=device),
      'edges':edges,
      'iw_ids':torch.as_tensor(iw_ids.astype(np.int64),device=device),
      'last_mask':torch.as_tensor(last_mask.astype(np.int64),device=device),
      'NX':torch.as_tensor(NX.astype(np.int64),device=device),
      'NX_rep':torch.as_tensor(NX_rep.astype(np.int64),device=device),
      'mol_ids_rep':torch.as_tensor(mol_ids_rep.astype(np.int64),device=device),
      'action_0':torch.as_tensor(action_0.astype(np.int64),device=device),
      'actions':torch.as_tensor(actions.astype(np.int64),device=device),
      'log_p_sigma':torch.as_tensor(log_p.astype(np.float32),device=device),
      'tox_class':torch.as_tensor(tox.astype(np.int64),device=device),
      'graph_to_rnn':torch.as_tensor(graph_to_rnn,device=device),
      'rnn_to_graph':torch.as_tensor(rnn_to_graph,device=device),
      'NX_cum':torch.as_tensor(np.cumsum(NX),dtype=torch.long,device=device),
      'batch_size':B, 'iw_size':k,
    }
    assert batch['tox_class'].shape==(B,),batch['tox_class'].shape
    assert batch['log_p_sigma'].shape==(B*k,)
    assert batch['action_0'].shape==(B*k,)
    assert batch['actions'].shape[0]==batch['NX'].shape[0]
    return batch

def make_batches(rows, epoch, batch_size=BATCH_SIZE, seed=SEED):
    # Length-bucketed sampler as in original CDGCN/CTDDG (limits memory for variable graph sizes).
    # Fully reproducible batch order for each epoch; final <B records are dropped to match original.
    indices=np.argsort([len(x) for x in rows]).tolist()
    n=len(indices)//batch_size
    if n==0:raise ValueError('Need at least batch_size molecules')
    rng=random.Random(seed+epoch)
    buckets=[indices[i*n:(i+1)*n] for i in range(batch_size)]
    for bucket in buckets:rng.shuffle(bucket)
    batches=[tuple(z) for z in zip(*buckets)]
    rng.shuffle(batches)
    return batches

In [ ]:
def sum_segments(x, seg_id, count):
    result=x.new_zeros((count,)+x.shape[1:])
    return result.index_add(0,seg_id,x)

class LinearBN(nn.Module):
    def __init__(self, fin, fout):
        super().__init__()
        self.linear=nn.Linear(fin,fout,bias=False)
        self.bn=nn.BatchNorm1d(fout)
    def forward(self,x):return self.bn(self.linear(x))

class GraphConv(nn.Module):
    """CDGCN GC with one self term + one matrix per bond/distance relation."""
    def __init__(self, fin, fout, n_relations):
        super().__init__()
        self.W=nn.Parameter(torch.empty(fin*(1+n_relations),fout))
        nn.init.xavier_uniform_(self.W)
        self.n_relations=n_relations

    def forward(self,X,edge_lists):
        assert len(edge_lists)==self.n_relations
        terms=[X]
        for edge in edge_lists:
            if edge.numel()==0:terms.append(torch.zeros_like(X))
            else:
                terms.append(sum_segments(X[edge[1]],edge[0],X.shape[0]))
        return torch.cat(terms,dim=-1) @ self.W

class GraphBackbone(nn.Module):
    def __init__(self,n_atom,n_bond,D,F_e,F_h,F_skip,F_c):
        super().__init__()
        self.atom_embedding=nn.Embedding(n_atom,F_e)
        self.mask_embedding=nn.Embedding(3,F_e)
        self.convs=nn.ModuleList()
        self.pre_bns=nn.ModuleList()
        for i,fout in enumerate(F_h):
            fin=F_e if i==0 else F_h[i-1]
            self.pre_bns.append(nn.Identity() if i==0 else nn.BatchNorm1d(fin))
            self.convs.append(GraphConv(fin,fout,n_bond+D))
        self.bn_skip=nn.BatchNorm1d(sum(F_h))
        self.skip=LinearBN(sum(F_h),F_skip)
        self.dense=nn.ModuleList()
        for fin,fout in zip([F_skip]+F_c[:-1],F_c):self.dense.append(LinearBN(fin,fout))

    def forward(self,b):
        X=self.atom_embedding(b['X'])+self.mask_embedding(b['last_mask'])
        layer_out=[]
        for conv,bn in zip(self.convs,self.pre_bns):
            X=conv(F.relu(bn(X)) if not isinstance(bn,nn.Identity) else X,b['edges'])
            layer_out.append(X)
        X=F.relu(self.skip(F.relu(self.bn_skip(torch.cat(layer_out,dim=-1)))))
        for dense in self.dense:X=dense(X)
        return X

class Policy(nn.Module):
    def __init__(self,fin,hidden,n_atom,n_bond):
        super().__init__()
        self.n_atom=n_atom;self.n_bond=n_bond
        self.linear_h=LinearBN(fin*2,hidden)
        self.linear_h_t=LinearBN(fin,hidden)
        self.linear_x=nn.Linear(hidden,n_bond*(n_atom+1))
        self.linear_x_t=nn.Linear(hidden,1)

    def forward(self,X,b,X_end):
        nx_rep=b['NX_rep'];n_step=b['NX'].shape[0]
        node_state=torch.cat([X,X_end[nx_rep]],dim=-1)
        node_logits=self.linear_x(F.relu(self.linear_h(node_state)))
        end_logits=self.linear_x_t(F.relu(self.linear_h_t(X_end)))
        # Stable per-graph softmax over every (node, append/connect action) AND end.
        per_node_max=node_logits.max(dim=-1).values
        group_max=end_logits[:,0].clone()
        group_max=group_max.scatter_reduce(0,nx_rep,per_node_max,reduce='amax',include_self=True)
        node_weight=torch.exp(node_logits-group_max[nx_rep,None])
        end_weight=torch.exp(end_logits-group_max[:,None])
        denom=sum_segments(node_weight.sum(-1,keepdim=True),nx_rep,n_step)+end_weight
        soft_node=node_weight/denom[nx_rep]
        soft_end=(end_weight/denom)[:,0]
        connect=soft_node[:,:self.n_bond]
        append=soft_node[:,self.n_bond:].reshape(-1,self.n_atom,self.n_bond)
        return append,connect,soft_end

class CTDDGPretrainer(nn.Module):
    def __init__(self,n_atom,n_bond,D=2,F_e=16,F_h=(32,64,128,128,256,256),
                 F_skip=256,F_c=(512,),F_policy=128,n_rnn=3):
        super().__init__()
        self.n_atom=n_atom;self.n_bond=n_bond
        self.backbone=GraphBackbone(n_atom,n_bond,D,F_e,list(F_h),F_skip,list(F_c))
        self.rnn=nn.GRU(input_size=2*F_c[-1],hidden_size=F_c[-1],num_layers=n_rnn,batch_first=True)
        self.policy=Policy(F_c[-1],F_policy,n_atom,n_bond)
        self.init_atom_logits=nn.Parameter(torch.zeros(n_atom))
        # Original MXNet notebook initializes trainable matrices with Xavier.
        # Match that policy rather than PyTorch's default Kaiming/embedding init.
        for module in self.modules():
            if isinstance(module,(nn.Linear,nn.Embedding)):
                nn.init.xavier_uniform_(module.weight)
                if getattr(module,'bias',None) is not None:nn.init.zeros_(module.bias)
            elif isinstance(module,nn.GRU):
                for name,param in module.named_parameters():
                    if 'weight' in name:nn.init.xavier_uniform_(param)
                    elif 'bias' in name:nn.init.zeros_(param)
            elif isinstance(module,nn.BatchNorm1d):
                nn.init.ones_(module.weight);nn.init.zeros_(module.bias)

    def per_molecule_loglik(self,b):
        B,K=b['batch_size'],b['iw_size']
        X=self.backbone(b)
        n_step=b['NX'].numel()
        X_avg=sum_segments(X,b['NX_rep'],n_step)/b['NX'][:,None].to(X.dtype)
        X_last=X[b['NX_cum']-1]
        features=torch.cat([X_avg,X_last],dim=-1)
        # Padded sequences; gathering only actual step outputs means padding cannot change valid outputs.
        padded=features[b['graph_to_rnn']].reshape(B*K,-1,features.shape[-1])
        rnn_out,_=self.rnn(padded)
        rnn_out=rnn_out.reshape(B,K,-1,rnn_out.shape[-1])
        idx=b['rnn_to_graph']
        X_end=rnn_out[idx[:,0],idx[:,1],idx[:,2]]
        append,connect,end=self.policy(X,b,X_end)
        act=b['actions'];act_type=act[:,0]
        eps=1e-10
        init_logprob=F.log_softmax(self.init_atom_logits,dim=0)
        init_ll=init_logprob[b['action_0']]
        # indices in actions refer to global offset into concatenated graph nodes
        append_log=torch.log(append[act[:,3],act[:,1],act[:,2]]+eps)
        connect_log=torch.log(connect[act[:,4],act[:,2]]+eps)
        end_log=torch.log(end+eps)
        action_ll=torch.where(act_type==0,append_log,torch.where(act_type==1,connect_log,end_log))
        path_ll=sum_segments(action_ll[:,None],b['iw_ids'],B*K)[:,0] + init_ll
        path_ll=path_ll.reshape(B,K)-b['log_p_sigma'].reshape(B,K)
        return torch.logsumexp(path_ll,dim=1)-math.log(K)

def toxicity_objective(ll, labels, lam=1.0):
    if ll.ndim!=1 or labels.shape!=ll.shape:raise ValueError(f'Expected aligned [B] tensors: {ll.shape}, {labels.shape}')
    if lam<0:raise ValueError('lambda must be nonnegative')
    return -(ll*(1.0-(1.0+lam)*labels.to(ll.dtype))).mean()

## 3. Add CDGCN conditioning without renaming pretrained parameters

`ConditionalGraphBackbone` inherits the same parameter paths `backbone.convs.*`, `backbone.dense.*`, `backbone.atom_embedding.*`, etc. and adds `backbone.protein_projections.*`. The conditional model also adds `init_head.*` for the first atom. **Transfer is strict:** every pretrained parameter and buffer must load with an identical name and shape; only these two conditioning groups may be new. Frozen ProSE embeddings themselves do not receive gradients.


In [ ]:
class ConditionalGraphBackbone(GraphBackbone):
    def __init__(self,n_atom,n_bond,D,F_e,F_h,F_skip,F_c,n_c):
        super().__init__(n_atom,n_bond,D,F_e,F_h,F_skip,F_c)
        self.protein_projections=nn.ModuleList([nn.Linear(n_c,width,bias=False) for width in F_h])
        for p in self.protein_projections:
            nn.init.xavier_uniform_(p.weight)

    def forward(self,b):
        X=self.atom_embedding(b['X'])+self.mask_embedding(b['last_mask'])
        c=b['c']
        molecule_for_node=b['mol_ids_rep']
        assert c.ndim==2 and c.shape[1]==N_C
        assert molecule_for_node.numel()==b['X'].numel()
        layers=[]
        for conv,bn,proj in zip(self.convs,self.pre_bns,self.protein_projections):
            g=conv(F.relu(bn(X)) if not isinstance(bn,nn.Identity) else X,b['edges'])
            X=g+proj(c)[molecule_for_node] # protein influences all intermediate GCN states
            layers.append(X)
        X=F.relu(self.skip(F.relu(self.bn_skip(torch.cat(layers,dim=-1)))))
        for dense in self.dense:X=dense(X)
        return X

class ConditionalCTDDG(CTDDGPretrainer):
    def __init__(self, model_config, n_c=N_C):
        super().__init__(**model_config)
        h=list(model_config['F_h'])
        self.backbone=ConditionalGraphBackbone(model_config['n_atom'],model_config['n_bond'],
                       model_config['D'],model_config['F_e'],h,model_config['F_skip'],
                       list(model_config['F_c']),n_c)
        nh=model_config['n_atom']*3
        self.init_head=nn.Sequential(nn.Linear(n_c,nh,bias=False),nn.BatchNorm1d(nh),
                                    nn.ReLU(),nn.Linear(nh,model_config['n_atom']))
        for module in self.init_head.modules():
            if isinstance(module,nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:nn.init.zeros_(module.bias)

    def per_molecule_loglik(self,b):
        B,K=b['batch_size'],b['iw_size']
        X=self.backbone(b)
        n_step=b['NX'].numel()
        X_avg=sum_segments(X,b['NX_rep'],n_step)/b['NX'][:,None].to(X.dtype)
        X_last=X[b['NX_cum']-1]
        features=torch.cat([X_avg,X_last],dim=-1)
        padded=features[b['graph_to_rnn']].reshape(B*K,-1,features.shape[-1])
        rnn_out,_=self.rnn(padded)
        rnn_out=rnn_out.reshape(B,K,-1,rnn_out.shape[-1])
        idx=b['rnn_to_graph']
        X_end=rnn_out[idx[:,0],idx[:,1],idx[:,2]]
        append,connect,end=self.policy(X,b,X_end)
        act=b['actions'];act_type=act[:,0]
        # Match CTDDG_pretraining_final(2).ipynb precisely (including eps in action log)
        eps=1e-10
        init_logprob=F.log_softmax(self.init_head(b['c']),dim=-1)
        init_ll=init_logprob.repeat_interleave(K,dim=0)[
            torch.arange(B*K,device=b['action_0'].device),b['action_0']]
        append_log=torch.log(append[act[:,3],act[:,1],act[:,2]]+eps)
        connect_log=torch.log(connect[act[:,4],act[:,2]]+eps)
        end_log=torch.log(end+eps)
        action_ll=torch.where(act_type==0,append_log,torch.where(act_type==1,connect_log,end_log))
        path_ll=sum_segments(action_ll[:,None],b['iw_ids'],B*K)[:,0]+init_ll
        path_ll=path_ll.reshape(B,K)-b['log_p_sigma'].reshape(B,K)
        return torch.logsumexp(path_ll,dim=1)-math.log(K)


def attach_proteins(data_rows,C):
    # Input `data_rows` is list of dicts from the prepared TSV, in exact collate order.
    strings=[f'{r["smiles"]} {int(r["ames"])}' for r in data_rows]
    b=collate_graph_records(strings,k=K_PATHS,alpha=ALPHA,device=DEVICE)
    index=np.array([int(r['protein_index']) for r in data_rows],dtype=np.int64)
    if (index<0).any() or (index>=len(C)).any():raise IndexError('Protein index outside HDF5 table')
    b['c']=torch.as_tensor(np.ascontiguousarray(C[index]),dtype=torch.float32,device=DEVICE)
    return b

# Toy test: a conditional checkpoint must be able to update new protein projectors.
assert not any(k.startswith('protein_projections.') for k in CTDDGPretrainer(3,4,F_h=(8,8),F_c=(16,),F_skip=12,F_policy=8,n_rnn=1).state_dict())
print('Conditional backbone and data-alignment adapter defined.')


## 4. Strict stage-1 transfer and preflight

A PyTorch state dict that merely loads is **not** sufficient: we compare parameter counts, vocabulary hash, model config and *all* shared tensors, then report the newly created conditioning weights. The trained model should be verified independently on generated molecules; this notebook only validates trainability and likelihood plumbing. The first-atom conditioning head starts at the pretrained atom marginal and learns protein dependence during fine-tuning.


In [ ]:
MODEL_CONFIG=pre_meta.get('model_config')
if not isinstance(MODEL_CONFIG,dict):raise RuntimeError('Stage-1 run_meta.json lacks model_config')
if MODEL_CONFIG['n_atom']!=get_mol_spec().num_atom_types:
    raise RuntimeError('Pretrain model expected a different number of atom types')
if MODEL_CONFIG['n_bond']!=get_mol_spec().num_bond_types or MODEL_CONFIG['D']!=RECEPTIVE_FIELD:
    raise RuntimeError('Bond/receptive-field geometry differs from pretraining')
# Trusted own checkpoint only: torch.load deserializes Python objects.
source=torch.load(PRETRAIN_CHECKPOINT,map_location='cpu',weights_only=False)
if not isinstance(source,dict) or not isinstance(source.get('model'),dict):
    raise RuntimeError('Expected pretraining checkpoint with model state dict')
model=ConditionalCTDDG(MODEL_CONFIG,N_C)
missing,unexpected=model.load_state_dict(source['model'],strict=False)
allowed_missing_prefix=('backbone.protein_projections.','init_head.')
if unexpected or any(not key.startswith(allowed_missing_prefix) for key in missing):
    raise RuntimeError(f'Invalid stage-1 transfer: unexpected {unexpected[:10]}, missing {missing[:10]}')
base_keys=set(source['model'])
# PyTorch initializes a new BatchNorm num_batches_tracked buffer at zero
# without returning it in `missing`; explicitly allow only new condition keys.
model_keys=set(model.state_dict())
new_keys=model_keys-base_keys
if not base_keys.issubset(model_keys) or any(not k.startswith(allowed_missing_prefix) for k in new_keys):
    raise RuntimeError(f'Unexpected state-dict keys after transfer: {sorted(new_keys)[:10]}')
if any(not torch.equal(model.state_dict()[key].cpu(), source['model'][key].cpu()) for key in base_keys):
    raise RuntimeError('One or more shared pretrained tensors did not transfer exactly')
with torch.no_grad():
    # Match pretraining first-atom distribution exactly before any protein learning.
    model.init_head[-1].weight.zero_()
    model.init_head[-1].bias.copy_(F.log_softmax(model.init_atom_logits,dim=0))
model.to(DEVICE)
print('Transferred',len(base_keys),'pretrained tensors; initialized',len(missing),'conditioning tensors')
print('Stage-1 epoch/step in checkpoint:',source.get('step','UNKNOWN'))
print('Trainable parameters:',sum(p.numel() for p in model.parameters()))
print('Check training provenance and RF model selection caveats before trusting this checkpoint.')
del source


## 5. Deterministic data, step-zero evaluation and safeguards

The source `make_batches` is carried over unchanged. It constructs reproducible batches from sequence/SMILES length across each epoch; despite its name it does not produce perfectly homogeneous batches. **The held-out TEST split is never fed to the training loop.** Validation graph paths use a fixed seed and reset Python RNG afterwards, so resuming or validating does not change subsequent training path sampling.


In [ ]:
rows_train=splits['train'].to_dict('records')
rows_val=splits['val'].to_dict('records')
if SMOKE_MAX_TRAIN_PAIRS is not None:
    rng=random.Random(SEED);rows_train=rng.sample(rows_train,min(len(rows_train),SMOKE_MAX_TRAIN_PAIRS))
if VAL_MAX_PAIRS is not None and len(rows_val)>VAL_MAX_PAIRS:
    rng=random.Random(SEED+35);rows_val=rng.sample(rows_val,VAL_MAX_PAIRS)
if len(rows_train)<BATCH_SIZE:raise ValueError('Too few train pairs for BATCH_SIZE')
# Use a shape-preserving string list solely for the original make_batches helper.
def batch_plan(epoch):
    strings=[r['smiles'] for r in rows_train]
    return make_batches(strings,epoch,batch_size=BATCH_SIZE,seed=SEED)

@torch.no_grad()
def evaluate_rows(model,rows,C,eval_seed=1729):
    was_train=model.training;py_state=random.getstate()
    model.eval();random.seed(eval_seed)
    sums={0:0.0,1:0.0};counts={0:0,1:0}
    total=0.0;weighted=0.0;n_total=0
    try:
        for off in range(0,len(rows),BATCH_SIZE):
            part=rows[off:off+BATCH_SIZE]
            b=attach_proteins(part,C)
            ll=model.per_molecule_loglik(b)
            if not torch.isfinite(ll).all():raise FloatingPointError('Nonfinite validation log-likelihood')
            tox=b['tox_class']
            loss=(toxicity_objective(ll,tox,TOX_LAMBDA) if OBJECTIVE=='ctddg_eq8' else -ll.mean())
            weighted+=float(loss)*len(part);total+=float(ll.sum());n_total+=len(part)
            for cls in (0,1):
                mask=tox==cls
                sums[cls]+=float(ll[mask].sum());counts[cls]+=int(mask.sum())
    finally:
        random.setstate(py_state);model.train(was_train)
    if not counts[0]:raise RuntimeError('Validation contains no Ames-negative ligands; cannot select checkpoint')
    return {'weighted_loss':weighted/n_total,'nll':-total/n_total,
            'll_nontoxic':sums[0]/counts[0],
            'll_toxic':sums[1]/counts[1] if counts[1] else None,
            'n_nontoxic':counts[0],'n_toxic':counts[1],'n_pairs':n_total,
            'selection_metric':-sums[0]/counts[0]}

print('Batches per epoch:',len(batch_plan(0)))
print('Preflight batch:')
b=attach_proteins(rows_train[:BATCH_SIZE],C_train)
assert b['c'].shape==(BATCH_SIZE,N_C) and b['mol_ids_rep'].max().item()<BATCH_SIZE
model.eval()
with torch.no_grad():
    ll=model.per_molecule_loglik(b)
    assert ll.shape==(BATCH_SIZE,) and torch.isfinite(ll).all()
    print('  l bounds:',ll.detach().cpu().numpy().round(2).tolist(), 'class 1 count:',int(b['tox_class'].sum()))
# Strict unit check that the loss reverses ONLY toxic examples.
x=torch.tensor([-2.,-3.],requires_grad=True)
y=toxicity_objective(x,torch.tensor([0,1]),1.)
y.backward()
assert torch.allclose(x.grad,torch.tensor([-0.5,0.5]))
# Finetune must update its new conditioning path (microbatch, no optimizer step).
model.train();model.zero_grad(set_to_none=True)
loss=toxicity_objective(model.per_molecule_loglik(b),b['tox_class'],TOX_LAMBDA) if OBJECTIVE=='ctddg_eq8' else -model.per_molecule_loglik(b).mean()
if not torch.isfinite(loss):raise FloatingPointError('Nonfinite preflight train-mode loss')
loss.backward()
if not any(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum()>0 for p in model.backbone.protein_projections.parameters()):
    raise RuntimeError('Protein conditioning has no gradient')
print('PASS: full conditional forward/backward, label signs, protein embedding gradients.')
model.zero_grad(set_to_none=True)
step_zero=evaluate_rows(model,rows_val,C_train)
print('Step-zero validation:',json.dumps(step_zero,indent=2))
if step_zero['ll_toxic'] is not None:
    print('Step-zero toxicity log-likelihood separation:',round(step_zero['ll_nontoxic']-step_zero['ll_toxic'],2))


## 6. Reproducible, **resumable** fine-tuning

**Resume semantics:** `ckpt_last.pt` stores **model, optimizer, RNG (Python/NumPy/Torch/CUDA), global optimizer step, epoch, next batch index, best validation metric and patience**, plus hashes of all data/config and the pretraining checkpoint. `make_batches` is deterministic per epoch, so resuming at batch *k* does not repeat or skip updates. Atomic checkpoint writes rotate the last good save to `.prev`, and Ctrl+C/Slurm SIGTERM requests a checkpoint after the current iteration. A hard kill can lose at most the interval since the last save; the saved state is never partially written. Use a different `RUN_DIR` for a changed experiment.

**Metric:** choose the lowest *negative mean log-likelihood on Ames-negative validation ligands*, not the signed Eq. 8 loss (unbounded below) or all-molecule NLL (rises if toxicity suppression works). This is only a proxy; evaluate generated toxicity/validity and target awareness separately before making conclusions.

**Crash log recovery:** if stdout/JSONL contained records beyond the last committed checkpoint, they are archived with an `.orphaned.<timestamp>` suffix and removed from the active log before resuming. This avoids duplicate step numbers after a hard kill.

In [ ]:
ckpt_last=RUN_DIR/'ckpt_last.pt'
ckpt_best=RUN_DIR/'ckpt_best.pt'
train_log=RUN_DIR/'train_log.jsonl'
val_log=RUN_DIR/'val_log.jsonl'
run_meta=RUN_DIR/'run_meta.json'

provenance={'schema_version':2,'architecture':'ConditionalCTDDG-compatible-with-CTDDG_pretraining_final(2)',
    'dataset_manifest_sha256':file_hash(DATA_DIR/'dataset_manifest.json'),
    'pretrain_checkpoint_sha256':file_hash(PRETRAIN_CHECKPOINT),
    'pretrain_run_meta_sha256':file_hash(PRETRAIN_META),'atom_types_sha256':file_hash(ATOM_TYPES_FILE),
    'model_config':MODEL_CONFIG,'n_c':N_C,'seed':SEED,'batch_size':BATCH_SIZE,
    'k_paths':K_PATHS,'alpha':ALPHA,'objective':OBJECTIVE,'tox_lambda':TOX_LAMBDA,
    'max_iter_per_mol':MAX_ITER_PER_MOL,'lr':LEARNING_RATE,'lr_decay':LR_DECAY,
    'lr_decay_step':LR_DECAY_STEP,'grad_clip':GRAD_CLIP,'patience':PATIENCE,
    'min_delta':MIN_DELTA,'val_max_pairs':VAL_MAX_PAIRS,
    'val_every_epochs':VAL_EVERY_EPOCHS,
    'smoke_max_train_pairs':SMOKE_MAX_TRAIN_PAIRS,'train_pairs':len(rows_train),
    'val_pairs':len(rows_val),'batches_per_epoch':len(batch_plan(0))}

def atomic_save(payload,path):
    tmp=path.with_name(path.name+f'.tmp.{os.getpid()}')
    with tmp.open('wb') as h:
        torch.save(payload,h);h.flush();os.fsync(h.fileno())
    if path.exists():os.replace(path,path.with_name(path.name+'.prev'))
    os.replace(tmp,path)

def load_good_checkpoint(path):
    for candidate in (path,path.with_name(path.name+'.prev')):
        if candidate.is_file():
            try:
                state=torch.load(candidate,map_location='cpu',weights_only=False) # only trusted own files
                if isinstance(state,dict) and 'model' in state and 'optimizer' in state:return state,candidate
            except Exception as exc:print('Unreadable checkpoint:',candidate,exc)
    return None,None

def get_rng():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def set_rng(saved):
    random.setstate(saved['python']);np.random.set_state(saved['numpy'])
    torch.set_rng_state(saved['torch'])
    if torch.cuda.is_available() and saved['cuda'] is not None:torch.cuda.set_rng_state_all(saved['cuda'])

def write_log(path,record):
    with path.open('a') as g:g.write(json.dumps(record,allow_nan=False)+'\n')

def reconcile_log(path, max_step, max_epoch=None):
    """Keep logs consistent with the last durable checkpoint after SIGKILL.

    Log entries newer than the saved optimizer state cannot be treated as a
    completed continuation. Preserve the original in an `.orphaned.*` backup.
    """
    if not path.is_file():return
    contents=path.read_text().splitlines(keepends=True)
    keep=[];orphan=[]
    for line in contents:
        try:
            entry=json.loads(line)
            step=int(entry.get('step',-1))
            epoch=int(entry.get('epoch',0))
            beyond=step>max_step or (max_epoch is not None and epoch>max_epoch)
        except Exception:
            beyond=True
        (orphan if beyond else keep).append(line)
    if orphan:
        backup=path.with_name(path.name+'.orphaned.'+time.strftime('%Y%m%d-%H%M%S'))
        shutil.copy2(path,backup)
        tmp=path.with_name(path.name+f'.tmp.{os.getpid()}')
        tmp.write_text(''.join(keep))
        os.replace(tmp,path)
        print(f'Recovered log {path.name}: archived {len(orphan)} orphaned lines at {backup.name}')

# Preflight can safely run without changing or creating a run directory.
resume_state=None
if RUN_DIR.exists():
    if run_meta.exists():
        old_meta=json.loads(run_meta.read_text())
        if old_meta!=provenance:
            raise RuntimeError('Existing fine-tuning run has different inputs/settings. Use a NEW RUN_DIR to start a different experiment.')
    elif ckpt_last.exists() or train_log.exists() or val_log.exists():
        raise RuntimeError('Run directory has artifacts but missing run_meta.json: refusing unsafe resume')
    if AUTO_RESUME:
        resume_state,loaded_path=load_good_checkpoint(ckpt_last)
        if resume_state is not None:print('Resumable state discovered:',loaded_path,'step',resume_state.get('global_step'))

optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
state={'epoch':0,'next_batch':0,'global_step':0,'best_metric':math.inf,'bad_epochs':0}
if resume_state is not None:
    if resume_state.get('provenance')!=provenance:
        raise RuntimeError('Checkpoint signature differs; cannot resume safely')
    model.load_state_dict(resume_state['model'],strict=True)
    optimizer.load_state_dict(resume_state['optimizer'])
    for s in optimizer.state.values():
        for key,value in s.items():
            if torch.is_tensor(value):s[key]=value.to(DEVICE)
    for key in state:
        state[key]=resume_state[key]
    set_rng(resume_state['rng'])
    # If a hard kill happened after a log write but before a durable checkpoint,
    # trim orphaned rows while archiving their exact bytes for auditability.
    reconcile_log(train_log,state['global_step'])
    reconcile_log(val_log,state['global_step'],max_epoch=state['epoch'])
    del resume_state

stop_requested={'flag':False,'signal':None}
def _signal_checkpoint_request(signum,frame):
    stop_requested.update(flag=True,signal=signal.Signals(signum).name)
    print('Signal received: save safely after current batch:',stop_requested['signal'],flush=True)

# Register only at training time (never changes handlers just by running preflight).
def save_state(best=False,reason='periodic'):
    snapshot={**state,'model':model.state_dict(),'optimizer':optimizer.state_dict(),
              'rng':get_rng(),'provenance':provenance,'reason':reason,
              'saved_at':time.strftime('%Y-%m-%d %H:%M:%S')}
    atomic_save(snapshot,ckpt_last)
    if best:
        atomic_save({'model':model.state_dict(),'provenance':provenance,
                     'global_step':state['global_step'],'epoch':state['epoch'],
                     'selection_metric':state['best_metric']},ckpt_best)

def train():
    if not RUN_TRAINING:
        print('RUN_TRAINING=False: preflight only, no optimizer steps taken')
        return
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()) and not run_meta.exists():
        raise RuntimeError('Non-empty RUN_DIR without provenance; use a NEW run directory')
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    if not run_meta.exists():run_meta.write_text(json.dumps(provenance,indent=2)+'\n')
    for ss in (signal.SIGINT,signal.SIGTERM):
        try:signal.signal(ss,_signal_checkpoint_request)
        except (ValueError,OSError):pass
    start_time=time.monotonic()
    while state['epoch']<MAX_EPOCHS and state['bad_epochs']<PATIENCE:
        ep=state['epoch'];plans=batch_plan(ep)
        assert len(plans)==provenance['batches_per_epoch']
        model.train()
        for bi in range(state['next_batch'],len(plans)):
            records=[rows_train[i] for i in plans[bi]]
            b=attach_proteins(records,C_train)
            optimizer.zero_grad(set_to_none=True)
            ll=model.per_molecule_loglik(b)
            if not torch.isfinite(ll).all():
                raise FloatingPointError(f'Nonfinite log-likelihood; last good checkpoint preserved at step {state["global_step"]}')
            if OBJECTIVE=='ctddg_eq8':loss=toxicity_objective(ll,b['tox_class'],TOX_LAMBDA)
            else:loss=-ll.mean() # true CDGCN all-positive baseline
            if not torch.isfinite(loss):raise FloatingPointError('Nonfinite loss (last checkpoint preserved)')
            loss.backward()
            grads=[p.grad for p in model.parameters() if p.grad is not None]
            if any(not torch.isfinite(g).all() for g in grads):
                raise FloatingPointError('Nonfinite gradient; last good checkpoint preserved')
            grad_norm=nn.utils.clip_grad_norm_(model.parameters(),float('inf'),error_if_nonfinite=True)
            nn.utils.clip_grad_value_(model.parameters(),GRAD_CLIP)
            optimizer.step()
            state['global_step']+=1
            state['next_batch']=bi+1
            if state['global_step']%LR_DECAY_STEP==0:
                for param in optimizer.param_groups:param['lr']*=(1-LR_DECAY)
            with torch.no_grad():
                tox=b['tox_class'];num0=int((tox==0).sum());num1=int((tox==1).sum())
                ln=float(ll[tox==0].mean()) if num0 else None
                lt=float(ll[tox==1].mean()) if num1 else None
            if state['global_step']==1 or state['global_step']%LOG_EVERY==0:
                line={'step':state['global_step'],'epoch':ep+1,'batch':bi+1,
                      'loss':float(loss.detach().item()),'ll_non':ln,'ll_toxic':lt,
                      'n_non':num0,'n_toxic':num1,
                      'grad_norm_pre_clip':float(grad_norm),
                      'lr':optimizer.param_groups[0]['lr'],
                      'elapsed_sec_this_session':round(time.monotonic()-start_time,2)}
                write_log(train_log,line)
                print('TRAIN',line,flush=True)
            if lt is not None and lt<TOX_LL_WARNING_THRESHOLD:
                print(f'[WARNING] Toxic log-likelihood {lt:.1f} below safety threshold {TOX_LL_WARNING_THRESHOLD}; negative Eq. 8 loss may represent divergence.',flush=True)
                if STOP_ON_TOX_LL_WARNING:
                    save_state(reason='toxic_branch_threshold')
                    raise RuntimeError('Toxic suppression threshold exceeded. Check generation quality before resuming.')
            if state['global_step']%SAVE_EVERY==0:
                save_state(reason='periodic')
            if stop_requested['flag']:
                save_state(reason=f'signal:{stop_requested["signal"]}')
                print('Interrupted and checkpointed. Rerun this notebook unchanged to resume.')
                return
        # Validate at stable epoch boundaries, after all training batches.
        state['epoch']=ep+1
        state['next_batch']=0
        if state['epoch']%VAL_EVERY_EPOCHS==0:
            metrics=evaluate_rows(model,rows_val,C_train)
            improved=(metrics['selection_metric'] < state['best_metric']-MIN_DELTA)
            if improved:
                state['best_metric']=metrics['selection_metric'];state['bad_epochs']=0
            else:
                state['bad_epochs']+=1
            metrics.update({'epoch':state['epoch'],'step':state['global_step'],
                            'best':state['best_metric'],'patience_counter':state['bad_epochs'],
                            'improved':improved})
            write_log(val_log,metrics)
            print('VALIDATION',metrics,flush=True)
            save_state(best=improved,reason='epoch_end')
        else:
            save_state(reason='epoch_end_no_validation')
    print('Fine-tuning finished at',state['global_step'],'steps;',state['epoch'],'epochs. Best selection metric:',state['best_metric'])

train()


## What to do after fine-tuning

1. Use `ckpt_best.pt` for **generation**, then compute RDKit validity/uniqueness, novelty, QED, SAS and RF-predicted Ames fraction; the latter alone cannot establish biological safety. Report a separate held-out protein-conditional evaluation, ideally with *per-protein* metrics, before docking.
2. Compare a **true CDGCN `OBJECTIVE='cdgcn_nll'` control**, an Eq. 8 run, and (if needed) stabilised alternatives in **distinct RUN_DIRs** with the same data and seeds.
3. Note that existing old `02_finetune_ctddg` generator checkpoints have **different parameter names and architecture**; do not use the old sample-generation notebook without adapting it to `ConditionalCTDDG` and auditing graph-action decoding. The current notebook deliberately does not pretend that sampling/evaluation has already been validated.
4. Slurm jobs: execute in a compute allocation; `SIGTERM` requests safe checkpointing, but very short termination grace periods or `SIGKILL` can still interrupt writes. Atomic latest/`.prev` saves protect earlier checkpoints. This notebook uses Python (not an unattended background service) and does not start training unless explicitly armed.

**Strict provenance gate:** The pretraining `run_meta.json` ChEMBL hash must match the exact file used by the BindingDB-exclusion audit. If it does not, investigate the actual stage-1 run; do not disable the gate and claim reproduction.